In [11]:
#%% PACKAGES
# Basics
import numpy as np                                                          
from math import pi                                                            
import warnings                                                              
import os    


from datetime import datetime, timedelta
from pandas import DataFrame
import geopandas as gpd

from sklearn.preprocessing import MinMaxScaler

# Visualization
import pandas as pd                                                          
import matplotlib.pyplot as plt                                                
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import numpy as np
import seaborn as sns; sns.set_theme(style='white')
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

In [12]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
IndexRaw = gpd.read_parquet(f'{output_step2_path}/step2_features.parquet')

attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")

In [13]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,attribute_source_path,initial_weight,impact_attribut,file_name,geometry_type,method,how,value_column,buffer_size,filter_column,filter_values,crs,save_format
0,Sécurité,accident,accident,True,accident,0.3,defavorable,OTC_ACCIDENTS-SHP/OTC_ACCIDENTS.shp,point,A,count,NaN,10,filtered,1,2056,parquet
1,Sécurité,traffic,zone_apaisee,True,vitesse,0.5,favorable,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
2,Sécurité,traffic,zone_pietonne,True,vitesse,1.0,favorable,OTC_ZONE_MODERATION_TRAFIC-SHP/OTC_ZONE_MODERA...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
3,Sécurité,traffic,vitesse,True,vitesse,0.6,defavorable,OTC_LIMITATIONS_VITESSE-SHP/OTC_LIMITATIONS_VI...,polygon,A,area_ratio,NaN,10,filtered,1,2056,parquet
4,Infrastructure,stationnement_genant,stationnement_genant,True,stationnement_genant,0.3,defavorable,SHP_FDP/FDP_STATIONNEMENT_GENANT_PIETON_CONTRA...,point,A,count,NaN,10,filtered,1,2056,parquet
5,Infrastructure,connectivite,connectivite,True,connectivite,0.7,favorable,NaN,line,A,sum,conn_branching_in_buffer,10,filtered,1,2056,parquet
6,Infrastructure,largeur_trottoir,largeur_trottoir,True,network_couche_OCT.shp,0.5,favorable,RP_final.shp,line,A,count,NaN,10,filtered,1,2056,parquet
7,Infrastructure,topographie,topographie,True,network_couche_OCT.shp,0.4,defavorable,RP_final.shp,line,A,sum,Pente,10,filtered,1,2056,parquet
8,Attractivité,eau,eau,True,eau,0.3,favorable,LCE_GRAPHE_EAU-SHP/LCE_GRAPHE_EAU.shp,line,A,count,NaN,10,filtered,1,2056,parquet
9,Attractivité,proximite,rez_actif,True,rez_actif,0.5,favorable,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,point,A,count,NaN,10,filtered,1,2056,parquet


In [14]:
print(attributs_info['attribute'].to_list())

['accident', 'zone_apaisee', 'zone_pietonne', 'vitesse', 'stationnement_genant', 'connectivite', 'largeur_trottoir', 'topographie', 'eau', 'rez_actif', 'tp', 'amenite', 'espaces_ouverts', 'temperature', 'canopee', 'bruit']


In [ ]:
#Delete where includes in index = False
attributs_info = attributs_info[attributs_info['include_in_index'] != False]
# Fill NULL
Indexv1 = IndexRaw.fillna(0)

# Create mapping for processed columns while keeping first 5 columns unchanged
attribute_mapping = {}
for _, row in attributs_info.iterrows():
    old_name = f"{row['attribute']}_{row['method']}_{row['buffer_size']}"
    new_name = row['attribute']
    attribute_mapping[old_name] = new_name

# Keep first 5 columns as is, rename the rest using the mapping
first_5_cols = IndexRaw.columns[:5].tolist()
Indexv1 = Indexv1.rename(columns=attribute_mapping)

# Debug info
print("First 5 columns:", first_5_cols)
print("Renamed columns:", [col for col in Indexv1.columns if col not in first_5_cols])

print('Crop outliers if necessary: --> see Walkability Amsterdam notebook')
Indexv2 = Indexv1.copy()

#Adding intervals (for factors where we have an interval of interest and below or above that interval the situation doesn't affect the walkability)
#Indexv2['stationnement_genant'] = Indexv1['stationnement_genant'].clip(upper=5) #more than 20 stationnement_genant in 10-meters-radius around segment centroid

# Create copy and normalize
Indexv3 = Indexv2.copy()
scaler = MinMaxScaler()

# Track changes during normalization
print("Min max normalization...")
for attribute in attributs_info['attribute']:
    if attribute in Indexv3.columns:
        # Normalize
        Indexv3.loc[:, attribute] = scaler.fit_transform(Indexv3[[attribute]]).round(4)
    else:
        print(f"Attribute '{attribute}' not found in Indexv3 columns.")
# Inverse columns (for factors that have a negative effect on walkability) depending on attributs_info.impact_attribut (favorable or defavorable)
print("Inverse columns where necessary...")
for _, row in attributs_info.iterrows():
    attribute_name = row['attribute']
    impact = row['impact_attribut']
    if attribute_name in Indexv3.columns:
        if impact == 'defavorable':
            Indexv3[attribute_name] = 1 - Indexv3[attribute_name]
            print(f"Inverted attribute: {attribute_name}")
    else:
        print(f"Attribute '{attribute_name}' not found in Indexv3 columns.")


First 5 columns: ['geometry', 'segment_id', 'length', 'accident_A_10', 'zone_apaisee_A_10']
Renamed columns: ['accident', 'zone_apaisee', 'zone_pietonne', 'vitesse', 'stationnement_genant', 'connectivite', 'largeur_trottoir', 'topographie', 'eau', 'rez_actif', 'tp', 'amenite', 'espaces_ouverts', 'temperature', 'canopee', 'bruit']
Crop outliers if necessary: --> see Walkability Amsterdam notebook
Min max normalization...
Inverse columns where necessary...
Inverted attribute: accident
Inverted attribute: vitesse
Inverted attribute: stationnement_genant
Inverted attribute: topographie
Inverted attribute: temperature
Inverted attribute: bruit


/var/folders/xd/q3z8hkl97ss7mdr3m6y7rn340000gn/T/ipykernel_75155/3811599163.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.0034 0.0535 0.2043 ... 0.0899 0.0129 0.0042]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  Indexv3.loc[:, attribute] = scaler.fit_transform(Indexv3[[attribute]]).round(4)


In [16]:
Indexv3

,geometry,segment_id,length,accident,zone_apaisee,zone_pietonne,vitesse,stationnement_genant,connectivite,largeur_trottoir,topographie,eau,rez_actif,tp,amenite,espaces_ouverts,temperature,canopee,bruit
0,"LINESTRING (6.22016 46.20288, 6.22023 46.20272)",000000,18.611510,1.0000,0.0,0.0000,0.8331,1.0000,0.0034,0.0870,0.9944,0.0,0.0000,0.1579,0.0000,0.0000,0.0683,0.0485,0.8945
1,"LINESTRING (6.20635 46.19641, 6.20656 46.19634)",000001,17.373877,0.9714,0.0,0.0000,0.8427,1.0000,0.0535,0.1739,0.9900,0.0,0.0000,0.2632,0.0000,0.1111,0.0350,0.0007,0.5556
2,"LINESTRING (6.14334 46.20439, 6.14354 46.2041,...",000002,50.000000,0.7714,0.0,0.0125,0.8432,0.9938,0.2043,0.3913,0.8874,0.0,0.0222,0.4737,0.0222,0.4444,0.0382,0.0000,0.5942
3,"LINESTRING (6.1436 46.20398, 6.14362 46.20391)",000003,7.198671,0.8571,0.0,0.0115,0.8424,0.9990,0.1351,0.3043,0.9608,0.0,0.0000,0.4211,0.0000,0.2222,1.0000,0.0000,0.6223
4,"LINESTRING (6.12495 46.18674, 6.12437 46.18679)",000004,45.125485,1.0000,0.0,0.1039,0.9168,1.0000,0.1317,0.4348,0.9112,0.0,0.0111,0.2105,0.0111,0.2222,0.0473,0.0039,0.7508
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134065,"LINESTRING (6.1299 46.16566, 6.12997 46.16563)",134065,6.136427,1.0000,0.0,0.0000,0.8331,1.0000,0.0110,0.0000,0.9851,0.0,0.0000,0.0000,0.0000,0.0000,1.0000,0.0002,0.9515
134066,"LINESTRING (6.11294 46.19517, 6.11291 46.19526)",134066,9.836936,1.0000,0.0,0.0000,0.8331,1.0000,0.0248,0.0000,0.9572,0.0,0.0000,0.2105,0.0000,0.0000,1.0000,0.0000,0.5556
134067,"LINESTRING (6.13432 46.18385, 6.13435 46.18382...",134067,9.957378,1.0000,0.0,0.0000,0.8331,1.0000,0.0899,0.0000,0.9873,0.0,0.0000,0.3158,0.0000,0.0000,1.0000,0.0072,0.8889
134068,"LINESTRING (6.10253 46.21532, 6.10261 46.21526)",134068,8.728223,1.0000,0.0,0.0000,0.8331,1.0000,0.0129,0.0000,0.9989,0.0,0.0000,0.1053,0.0000,0.0000,0.0309,0.0020,0.8889


In [17]:
# Calculate the Main Index Scores
Indexv4 = Indexv3

# Get weights from attributs_info and check validity
Zscore_weights = {}
for _, row in attributs_info[attributs_info.include_in_index == True].iterrows():
    attr = row['attribute']
    weight = row['initial_weight']
    if pd.isnull(weight):
        raise ValueError(f"Weight not specified for attribute '{attr}' (found NaN). Please specify a value between 0 and 1.")
    if not (0 <= weight <= 1):
        raise ValueError(f"Weight for attribute '{attr}' is {weight}, but must be between 0 and 1.")
    Zscore_weights[attr] = weight

print("Zscore_weights:", Zscore_weights)


# add zscore weights to the attributes_info dataframe
attributs_info['weights'] = attributs_info['attribute'].map(Zscore_weights)

# Function Sum-product to calculate sub-index score
def calculate_index(df, weights_dict):
    return sum(df[col] * weight for col, weight in weights_dict.items())

Indexv4['I-Zscore'] = calculate_index(Indexv4, Zscore_weights)
Indexv4['I-Zscore_unweighted'] = calculate_index(Indexv4, {key: 1 for key in Zscore_weights.keys()})

#Normalising Index
#Indexv4['Non-Scaled']=Indexv4['I-Zscore'].round(4)
Indexv4[['I-Zscore']] = scaler.fit_transform(Indexv4[['I-Zscore']]).round(4)
Indexv4[['I-Zscore_unweighted']] = scaler.fit_transform(Indexv4[['I-Zscore_unweighted']]).round(4)


# Attribut zone piétonne est prédominant
Indexv5 = Indexv4
max_score = Indexv4["I-Zscore"].max()

Indexv5["walkability_index"] = np.where(
    Indexv5["zone_pietonne"] > 0,
    max_score,               # assign max score if pedestrian zone
    Indexv5["I-Zscore"]           # else keep original
)


Zscore_weights: {'accident': 0.3, 'zone_apaisee': 0.5, 'zone_pietonne': 1.0, 'vitesse': 0.6, 'stationnement_genant': 0.3, 'connectivite': 0.7, 'largeur_trottoir': 0.5, 'topographie': 0.4, 'eau': 0.3, 'rez_actif': 0.5, 'tp': 0.6, 'amenite': 0.6, 'espaces_ouverts': 0.8, 'temperature': 0.6, 'canopee': 0.5, 'bruit': 0.4}


In [ ]:
# Calculate index for each class
Indexv6 = Indexv5


for cls in attributs_info['Class'].unique():
    # Select attributes belonging to this class and included in the index
    subset = attributs_info[
        (attributs_info['Class'] == cls) & 
        (attributs_info['include_in_index'])
    ]

    # Build the weight dictionary for this class only
    cls_weights = {row['attribute']: row['initial_weight'] for _, row in subset.iterrows()}

    # Compute weighted sub-index
    Indexv6[f"I-Zscore_{cls}"] = calculate_index(Indexv6, cls_weights)
    # Normalize the result
    Indexv6[[f"I-Zscore_{cls}"]] = scaler.fit_transform(Indexv6[[f"I-Zscore_{cls}"]]).round(4)

In [19]:
Indexv6.head()

,geometry,segment_id,length,accident,zone_apaisee,zone_pietonne,vitesse,stationnement_genant,connectivite,largeur_trottoir,topographie,eau,rez_actif,tp,amenite,espaces_ouverts,temperature,canopee,bruit,I-Zscore,I-Zscore_unweighted,walkability_index,I-Zscore_Sécurité,I-Zscore_Infrastructure,I-Zscore_Attractivité,I-Zscore_Agrément
0,"LINESTRING (6.22016 46.20288, 6.22023 46.20272)",000000,18.611510,1.0000,0.0,0.0000,0.8331,1.0000,0.0034,0.0870,0.9944,0.0,0.0000,0.1579,0.0000,0.0000,0.0683,0.0485,0.8945,0.2470,0.3842,0.2470,0.3212,0.3459,0.0773,0.2903
1,"LINESTRING (6.20635 46.19641, 6.20656 46.19634)",000001,17.373877,0.9714,0.0,0.0000,0.8427,1.0000,0.0535,0.1739,0.9900,0.0,0.0000,0.2632,0.0000,0.1111,0.0350,0.0007,0.5556,0.2626,0.3641,0.2626,0.3194,0.4271,0.2013,0.1653
2,"LINESTRING (6.14334 46.20439, 6.14354 46.2041,...",000002,50.000000,0.7714,0.0,0.0125,0.8432,0.9938,0.2043,0.3913,0.8874,0.0,0.0222,0.4737,0.0222,0.4444,0.0382,0.0000,0.5942,0.4489,0.5213,1.0000,0.2892,0.6084,0.5416,0.1772
3,"LINESTRING (6.1436 46.20398, 6.14362 46.20391)",000003,7.198671,0.8571,0.0,0.0115,0.8424,0.9990,0.1351,0.3043,0.9608,0.0,0.0000,0.4211,0.0000,0.2222,1.0000,0.0000,0.6223,0.5549,0.6731,1.0000,0.3047,0.5439,0.3510,0.5869
4,"LINESTRING (6.12495 46.18674, 6.12437 46.18679)",000004,45.125485,1.0000,0.0,0.1039,0.9168,1.0000,0.1317,0.4348,0.9112,0.0,0.0111,0.2105,0.0111,0.2222,0.0473,0.0039,0.7508,0.4184,0.5340,1.0000,0.4198,0.5897,0.2579,0.2260


In [20]:
# Save it
Indexv6.to_crs(target_crs).to_csv(f'{output_step3_path}/step3_index.csv', index = False)
Indexv6.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_index.parquet')
Indexv6.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_index.gpkg"), driver="GPKG")